# 10 — P2: Train Lipschitz-Margin ViT-Tiny on MNIST

**Plan 2 — Phase 1.3**: Train the *same* ViT-Tiny architecture as notebook 09, but with:
1. **Spectral normalization** on every `Linear` and `Conv2d` layer (bounds each layer's Lipschitz constant ≤ 1).
2. **Margin loss** (multi-class hinge) added to cross-entropy — pushes logit margins wide so the global Lipschitz pre-filter (Phase 6) certifies more samples.

Reference: LipShiFT (https://arxiv.org/abs/2503.14751, https://github.com/RohanMenon/LipShiFT). We do **not** replace MHSA with shift modules — the goal of Plan 2 is to verify the standard MHSA encoder block.

## Output
Checkpoint at `runs/vit_tiny_lipmargin/model.pt`. Target ≥92% test accuracy; small global Lipschitz bound (measured in Phase 2).

In [1]:
!pip install -q numpy pandas torch torchvision tqdm pyyaml

In [2]:
# ── Imports & seed ────────────────────────────────────────────────────────
from __future__ import annotations
import math, json, time, warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.nn.utils.parametrizations import spectral_norm
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

def set_seed(seed: int = 1234) -> None:
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(1234)
print(f'Device : {device}\nPyTorch: {torch.__version__}')

Device : cuda
PyTorch: 2.10.0+cu128


In [24]:
# ── Config ────────────────────────────────────────────────────────────────
CFG = dict(
    seed         = 1234,
    data_root    = '/tmp/mnist',
    run_dir      = r'C:\Users\manya\OneDrive\Desktop\THESIS\formal-verification\runs\vit_tiny_lipmargin',
    n_epochs     = 30,
    batch_size   = 128,
    lr           = 3e-4,
    weight_decay = 0.01,
    # Architecture (must match notebook 09 exactly)
    img_size     = 28,
    patch_size   = 4,
    embed_dim    = 64,
    num_heads    = 2,
    num_layers   = 2,
    mlp_ratio    = 2,
    eps_rms      = 1e-6,
    # Lipschitz-margin training
    margin_kappa = 1.0,    # hinge margin κ — wider = stronger Lipschitz incentive
    lambda_marg  = 0.5,    # weight on margin term (CE has weight 1.0)
    sn_n_iter    = 1,      # power-iteration steps for spectral_norm
)
Path(CFG['run_dir']).mkdir(parents=True, exist_ok=True)
print(json.dumps(CFG, indent=2))

{
  "seed": 1234,
  "data_root": "/tmp/mnist",
  "run_dir": "C:\\Users\\manya\\OneDrive\\Desktop\\THESIS\\formal-verification\\runs\\vit_tiny_lipmargin",
  "n_epochs": 30,
  "batch_size": 128,
  "lr": 0.0003,
  "weight_decay": 0.01,
  "img_size": 28,
  "patch_size": 4,
  "embed_dim": 64,
  "num_heads": 2,
  "num_layers": 2,
  "mlp_ratio": 2,
  "eps_rms": 1e-06,
  "margin_kappa": 1.0,
  "lambda_marg": 0.5,
  "sn_n_iter": 1
}


In [25]:
# ── Data ──────────────────────────────────────────────────────────────────
def get_mnist_loaders(batch_size=128, data_root='/tmp/mnist'):
    tf = transforms.ToTensor()
    train_ds = torchvision.datasets.MNIST(data_root, train=True,  download=True, transform=tf)
    test_ds  = torchvision.datasets.MNIST(data_root, train=False, download=True, transform=tf)
    kw = dict(num_workers=0, pin_memory=False)
    return (
        torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **kw),
        torch.utils.data.DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw),
    )

train_loader, test_loader = get_mnist_loaders(CFG['batch_size'], CFG['data_root'])
print(f'Train: {len(train_loader.dataset):,}  Test: {len(test_loader.dataset):,}')

Train: 60,000  Test: 10,000


In [26]:
# ── ViT-Tiny model (identical architecture to notebook 09) ────────────────
#
# Spectral normalization is applied AFTER construction (cell below) so the
# class definition stays one-to-one with notebook 09. Phase 2 / 4 / 5 phases
# can load either checkpoint into the same class.

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.dim, self.eps = dim, eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)


class PatchEmbed(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, embed_dim=64):
        super().__init__()
        assert img_size % patch_size == 0
        self.img_size, self.patch_size = img_size, patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class MHSA(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.scale    = self.head_dim ** -0.5
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=True)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=True)
    def forward(self, x):
        B, N, C = x.shape
        H, D = self.num_heads, self.head_dim
        q = self.W_q(x).view(B, N, H, D).transpose(1, 2)
        k = self.W_k(x).view(B, N, H, D).transpose(1, 2)
        v = self.W_v(x).view(B, N, H, D).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn   = F.softmax(scores, dim=-1)
        out    = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, C)
        return self.W_o(out)


class MLPBlock(nn.Module):
    def __init__(self, embed_dim: int, mlp_ratio: int = 2):
        super().__init__()
        hidden = embed_dim * mlp_ratio
        self.fc1 = nn.Linear(embed_dim, hidden)
        self.fc2 = nn.Linear(hidden, embed_dim)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: int = 2,
                 eps_rms: float = 1e-6):
        super().__init__()
        self.norm1 = RMSNorm(embed_dim, eps_rms)
        self.attn  = MHSA(embed_dim, num_heads)
        self.norm2 = RMSNorm(embed_dim, eps_rms)
        self.mlp   = MLPBlock(embed_dim, mlp_ratio)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class ViTTiny(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, num_classes=10,
                 embed_dim=64, num_heads=2, num_layers=2, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.cfg = dict(img_size=img_size, patch_size=patch_size,
                        in_channels=in_channels, num_classes=num_classes,
                        embed_dim=embed_dim, num_heads=num_heads,
                        num_layers=num_layers, mlp_ratio=mlp_ratio, eps_rms=eps_rms)
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.patch_embed.n_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, eps_rms)
            for _ in range(num_layers)
        ])
        self.norm = RMSNorm(embed_dim, eps_rms)
        self.head = nn.Linear(embed_dim, num_classes)
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        for blk in self.blocks: x = blk(x)
        x = self.norm(x).mean(dim=1)
        return self.head(x)

print('Model class defined')

Model class defined


In [27]:
# ── Apply spectral normalization to every Linear and Conv2d layer ─────────
#
# Strategy: caps σ_max(W) ≤ 1 for every layer that contributes a singular
# value to the global Lipschitz bound. The patch_embed conv, all Q/K/V/O
# attention projections, both MLP linears, and the classifier head are
# wrapped. RMSNorm is intentionally NOT touched (its Lipschitz is bounded
# by ||γ||∞ · √(D/ε_rms); we let γ float and account for it in Phase 2).

def apply_spectral_norm(model: nn.Module, n_iter: int = 1) -> int:
    n = 0
    for parent in model.modules():
        for child_name, child in list(parent.named_children()):
            if isinstance(child, (nn.Linear, nn.Conv2d)):
                setattr(parent, child_name,
                        spectral_norm(child, n_power_iterations=n_iter))
                n += 1
    return n

set_seed(CFG['seed'])
model = ViTTiny(
    img_size=CFG['img_size'], patch_size=CFG['patch_size'],
    embed_dim=CFG['embed_dim'], num_heads=CFG['num_heads'],
    num_layers=CFG['num_layers'], mlp_ratio=CFG['mlp_ratio'],
    eps_rms=CFG['eps_rms'],
).to(device)
n_wrapped = apply_spectral_norm(model, CFG['sn_n_iter'])
n_params  = sum(p.numel() for p in model.parameters())
print(f'Spectral-norm wrapped {n_wrapped} layers')
print(f'Total parameters     : {n_params:,}')

# Smoke check: forward pass works and gives finite logits
with torch.no_grad():
    _x = torch.zeros(2, 1, 28, 28, device=device)
    _y = model(_x)
    print(f'Output shape={tuple(_y.shape)}  finite={torch.isfinite(_y).all().item()}')

Spectral-norm wrapped 14 layers
Total parameters     : 71,370
Output shape=(2, 10)  finite=True


In [28]:
# ── Loss: CE + multi-class hinge margin ───────────────────────────────────
#
# margin_loss = mean_i  max(0,  max_{c≠y_i} z_c  -  z_{y_i}  +  κ )
#
# Wider margins (high κ, large λ) → model logits are less sensitive to
# input perturbations of size ε for any given Lipschitz constant L,
# so more samples pass the Phase 6 Lipschitz pre-filter
# (margin > L · ε · √D).

def margin_loss(logits: torch.Tensor, y: torch.Tensor, kappa: float) -> torch.Tensor:
    B, C = logits.shape
    z_y      = logits.gather(1, y.unsqueeze(1)).squeeze(1)            # (B,)
    masked   = logits.clone()
    masked.scatter_(1, y.unsqueeze(1), float('-inf'))
    z_other  = masked.max(dim=1).values                               # (B,)
    return F.relu(z_other - z_y + kappa).mean()


def total_loss(logits: torch.Tensor, y: torch.Tensor,
               kappa: float, lam: float):
    ce  = F.cross_entropy(logits, y)
    mar = margin_loss(logits, y, kappa)
    return ce + lam * mar, ce.detach(), mar.detach()

In [29]:
# ── Train loop ────────────────────────────────────────────────────────────
@torch.no_grad()
def eval_acc_and_margin(model, loader, device):
    model.eval()
    correct = total = 0
    margins = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        z = model(x)
        correct += (z.argmax(1) == y).sum().item()
        total   += len(y)
        z_y     = z.gather(1, y.unsqueeze(1)).squeeze(1)
        m       = z.clone(); m.scatter_(1, y.unsqueeze(1), float('-inf'))
        margins.append((z_y - m.max(dim=1).values).cpu())
    margins = torch.cat(margins)
    return correct / total, margins.mean().item(), margins.median().item()


opt   = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['n_epochs'])

best_acc, best_state = 0.0, None
history = []
for ep in range(1, CFG['n_epochs'] + 1):
    t0 = time.time()
    model.train()
    sum_loss = sum_ce = sum_mar = n = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        z = model(x)
        loss, ce, mar = total_loss(z, y, CFG['margin_kappa'], CFG['lambda_marg'])
        loss.backward()
        opt.step()
        bs = len(y)
        sum_loss += loss.item() * bs
        sum_ce   += ce.item()   * bs
        sum_mar  += mar.item()  * bs
        n        += bs
    sched.step()

    test_acc, mean_margin, median_margin = eval_acc_and_margin(model, test_loader, device)
    history.append(dict(epoch=ep,
                        train_loss=sum_loss/n, train_ce=sum_ce/n, train_margin=sum_mar/n,
                        test_acc=test_acc, test_margin_mean=mean_margin,
                        test_margin_median=median_margin,
                        lr=opt.param_groups[0]['lr']))
    if test_acc > best_acc:
        best_acc, best_state = test_acc, deepcopy(model.state_dict())
    print(f'ep {ep:02d}/{CFG["n_epochs"]:02d}  '
          f'loss={sum_loss/n:.4f} (ce={sum_ce/n:.4f} mar={sum_mar/n:.4f})  '
          f'test_acc={test_acc:.4f}  margin_med={median_margin:.3f}  '
          f'({time.time()-t0:.0f}s)')

model.load_state_dict(best_state)
print(f'\nBest test accuracy: {best_acc:.4f}')
print(f'Target ≥ 0.92 — {"PASS" if best_acc >= 0.92 else "BELOW TARGET"}')

ep 01/30  loss=0.9482 (ce=0.6839 mar=0.5285)  test_acc=0.8823  margin_med=3.062  (16s)
ep 02/30  loss=0.3932 (ce=0.2735 mar=0.2392)  test_acc=0.9376  margin_med=4.095  (15s)
ep 03/30  loss=0.2818 (ce=0.1957 mar=0.1722)  test_acc=0.9534  margin_med=4.697  (16s)
ep 04/30  loss=0.2286 (ce=0.1589 mar=0.1395)  test_acc=0.9609  margin_med=5.018  (16s)
ep 05/30  loss=0.1976 (ce=0.1371 mar=0.1210)  test_acc=0.9689  margin_med=5.345  (15s)
ep 06/30  loss=0.1690 (ce=0.1174 mar=0.1033)  test_acc=0.9704  margin_med=5.492  (15s)
ep 07/30  loss=0.1547 (ce=0.1072 mar=0.0950)  test_acc=0.9670  margin_med=5.376  (15s)
ep 08/30  loss=0.1366 (ce=0.0949 mar=0.0835)  test_acc=0.9679  margin_med=5.769  (15s)
ep 09/30  loss=0.1323 (ce=0.0915 mar=0.0816)  test_acc=0.9700  margin_med=6.060  (16s)
ep 10/30  loss=0.1138 (ce=0.0792 mar=0.0692)  test_acc=0.9772  margin_med=6.045  (15s)
ep 11/30  loss=0.1058 (ce=0.0735 mar=0.0647)  test_acc=0.9732  margin_med=6.094  (15s)
ep 12/30  loss=0.0936 (ce=0.0651 mar=0.0570

In [31]:
# ── Save checkpoint ───────────────────────────────────────────────────────
#
# Spectral-norm wraps each Linear/Conv with a parametrization. The state_dict
# stores both the original weight (`weight_orig`) and the SVD power-iteration
# vectors. Saving as-is means downstream notebooks must re-wrap before load.
#
# Convenience: also save a "materialized" state_dict (post-SN weights baked
# in, no parametrization) so Phase 2/4/5 can load into the plain ViTTiny
# class from notebook 09 without any spectral_norm import.

from torch.nn.utils.parametrize import remove_parametrizations

# Materialize: remove SN parametrizations on a copy and grab its state_dict
model_mat = deepcopy(model)
for parent in model_mat.modules():
    for child_name, child in list(parent.named_children()):
        if hasattr(child, 'parametrizations') and 'weight' in child.parametrizations:
            remove_parametrizations(child, 'weight', leave_parametrized=True)

# run_dir = r'C:\Users\manya\OneDrive\Desktop\THESIS\formal-verification\runs\vit_tiny_lipmargin'

ckpt_path = Path(CFG['run_dir']) / 'model.pt'
torch.save({
    'state_dict':            model.state_dict(),       # SN-parametrized
    'state_dict_materialized': model_mat.state_dict(), # plain weights
    'cfg':       model.cfg,
    'best_acc':  best_acc,
    'history':   history,
    'training':  {k: CFG[k] for k in ('seed','n_epochs','batch_size','lr','weight_decay',
                                      'margin_kappa','lambda_marg','sn_n_iter')},
    'spectral_norm_applied': True,
}, ckpt_path)
print(f'Saved → {ckpt_path}  ({ckpt_path.stat().st_size/1e3:.1f} KB)')
print(f'Best test acc: {best_acc:.4f}')

# # Sanity: materialized model gives identical outputs to SN-wrapped model
# with torch.no_grad():
#     _x = torch.randn(4, 1, 28, 28, device=device)
#     _a = model(_x); _b = model_mat(_x)
#     diff = (_a - _b).abs().max().item()
#     print(f'SN vs materialized max-abs diff: {diff:.2e}  ({"OK" if diff < 1e-4 else "MISMATCH"})')

AttributeError: 'ParametrizedLinear' object has no attribute 'weight'

In [10]:
pwd

'/content'

In [11]:
cd ../

/


In [12]:
pwd

'/'

In [13]:
ls

bin@                        lib32@                    root/
boot/                       lib64@                    run/
content/                    libx32@                   sbin@
cuda-keyring_1.1-1_all.deb  media/                    srv/
datalab/                    mnt/                      sys/
dev/                        NGC-DL-CONTAINER-LICENSE  tmp/
etc/                        opt/                      tools/
home/                       proc/                     usr/
kaggle/                     python-apt/               var/
lib@                        python-apt.tar.xz*
